# Figure S2: Additional Cases

- Panels: `FigS2a`, `FigS2b`, `FigS2c`, `FigS2d`
- Role: supplementary qualitative cases beyond the main-text representative panels
- Single-perturbation case: `Adamson / PTDSS1+ctrl / split 4`
- Combo case: `Norman / DUSP9+IGDCC3 / split 5`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis._result_adapter import load_payload_item

apply_gears_paper_style(font_scale=1.0)
MODEL_ORDER = ['trishift_nearest', 'genepert', 'scgpt', 'biolord']
MODEL_LABELS = {'trishift_nearest':'TriShift','genepert':'GenePert','scgpt':'scGPT','biolord':'biolord'}
MODEL_COLORS = {'TriShift':'#9FD9D3','GenePert':'#87A8D8','scGPT':'#DDD3C8','biolord':'#F0806A','Truth':'#7F7F7F','Control':'#CFCFCF','Perturbed':'#7F7F7F'}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS2_AdditionalCases'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
S2AB_SINGLE_CASE = {'dataset': 'adamson', 'split_id': 4, 'condition': 'PTDSS1+ctrl', 'gene': 'RPS29'}
S2CD_COMBO_CASE = {'dataset': 'norman', 'split_id': 5, 'condition': 'BPGM+ZBTB1', 'gene': 'HBZ'}

def _style_axis(ax):
    paper_style_axis(ax, grid_axis='y')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

def _top_genes_by_truth_delta(item, top_k=12):
    genes = np.asarray(item['gene_name_full']).astype(str)
    truth = np.asarray(item['Truth_full'], dtype=float)
    ctrl = np.asarray(item['Ctrl_full'], dtype=float)
    delta = truth.mean(axis=0) - ctrl.mean(axis=0)
    top_idx = np.argsort(-np.abs(delta))[:top_k]
    return genes[top_idx], delta[top_idx], top_idx

def _case_barplot(dataset, split_id, condition, out_name, top_k=12):
    items = {m: load_payload_item(dataset=dataset, model_name=m, split_id=split_id, condition=condition) for m in MODEL_ORDER}
    ref_item = items['trishift_nearest']
    genes, truth_delta, top_idx = _top_genes_by_truth_delta(ref_item, top_k=top_k)
    rows = []
    for gene, idx, delta in zip(genes, top_idx, truth_delta):
        rows.append({'gene': gene, 'model': 'Truth', 'delta': float(delta)})
        for model_name, item in items.items():
            pred = np.asarray(item['Pred_full'], dtype=float)
            ctrl = np.asarray(item['Ctrl_full'], dtype=float)
            model_genes = np.asarray(item['gene_name_full']).astype(str)
            model_idx = int(np.where(model_genes == gene)[0][0])
            pred_delta = pred[:, model_idx].mean() - ctrl[:, model_idx].mean()
            rows.append({'gene': gene, 'model': MODEL_LABELS[model_name], 'delta': float(pred_delta)})
    plot_df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(9.0, 5.2), dpi=220)
    sns.barplot(data=plot_df, x='gene', y='delta', hue='model', hue_order=['Truth','TriShift','GenePert','scGPT','biolord'], palette=MODEL_COLORS, errorbar=None, ax=ax)
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(0.5)
    ax.set_xlabel('')
    ax.set_ylabel('Change over control')
    ax.set_title(f'{condition} ({dataset.capitalize()}, split {split_id})', pad=12)
    ax.tick_params(axis='x', rotation=38)
    ax.legend(
        title='',
        frameon=False,
        ncol=5,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.56),
        borderaxespad=0.0,
        handlelength=1.6,
        columnspacing=1.2,
    )
    _style_axis(ax)
    fig.subplots_adjust(top=0.88, bottom=0.38)
    fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
    plot_df.to_csv(OUT_ROOT / out_name.replace('.png', '_values.csv'), index=False)
    plt.close(fig)

def _violin_case(dataset, split_id, condition, gene, out_name):
    rows = []
    base_item = load_payload_item(dataset=dataset, model_name='trishift_nearest', split_id=split_id, condition=condition)
    genes = np.asarray(base_item['gene_name_full']).astype(str)
    gene_idx = int(np.where(genes == gene)[0][0])
    for value in np.asarray(base_item['Ctrl_full'], dtype=float)[:, gene_idx]:
        rows.append({'group': 'Control', 'expression': float(value)})
    for value in np.asarray(base_item['Truth_full'], dtype=float)[:, gene_idx]:
        rows.append({'group': 'Perturbed', 'expression': float(value)})
    for model_name in MODEL_ORDER:
        item = load_payload_item(dataset=dataset, model_name=model_name, split_id=split_id, condition=condition)
        model_genes = np.asarray(item['gene_name_full']).astype(str)
        model_idx = int(np.where(model_genes == gene)[0][0])
        for value in np.asarray(item['Pred_full'], dtype=float)[:, model_idx]:
            rows.append({'group': MODEL_LABELS[model_name], 'expression': float(value)})
    plot_df = pd.DataFrame(rows)
    order = ['Control','Perturbed','TriShift','GenePert','scGPT','biolord']
    fig, ax = plt.subplots(figsize=(6.6, 4.4), dpi=240)
    sns.violinplot(data=plot_df, x='group', y='expression', hue='group', order=order, hue_order=order, palette=MODEL_COLORS, inner='quartile', cut=0, linewidth=0.8, dodge=False, legend=False, width=0.82, ax=ax)
    for coll in ax.collections:
        try:
            coll.set_edgecolor('black')
            coll.set_linewidth(0.8)
        except Exception:
            pass
    ax.set_xlabel('')
    ax.set_ylabel(f'{gene} expression')
    ax.set_title(f'{gene} | {condition}', pad=10)
    ax.tick_params(axis='x', labelrotation=28)
    _style_axis(ax)
    fig.tight_layout(rect=[0,0,1,0.96])
    fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
    plot_df.to_csv(OUT_ROOT / out_name.replace('.png', '_values.csv'), index=False)
    plt.close(fig)


In [2]:
_case_barplot(S2AB_SINGLE_CASE['dataset'], S2AB_SINGLE_CASE['split_id'], S2AB_SINGLE_CASE['condition'], 'figs2a_single_case_barplot.png', top_k=12)
_violin_case(S2AB_SINGLE_CASE['dataset'], S2AB_SINGLE_CASE['split_id'], S2AB_SINGLE_CASE['condition'], S2AB_SINGLE_CASE['gene'], 'figs2b_single_case_violin.png')
_case_barplot(S2CD_COMBO_CASE['dataset'], S2CD_COMBO_CASE['split_id'], S2CD_COMBO_CASE['condition'], 'figs2c_combo_case_barplot.png', top_k=12)
_violin_case(S2CD_COMBO_CASE['dataset'], S2CD_COMBO_CASE['split_id'], S2CD_COMBO_CASE['condition'], S2CD_COMBO_CASE['gene'], 'figs2d_combo_case_violin.png')
print(OUT_ROOT)


E:\CODE\trishift\artifacts\paper_figures\supp\FigS2_AdditionalCases
